# Exploração e Modelagem Inicial - Cibersegurança\n\nNotebook base para classificação de ataques de rede com NSL-KDD ou CICIDS2017, comparando Random Forest e SVM.

## 1) Carregamento dos dados\nDefina o caminho do dataset em `data/raw/` e ajuste separador/colunas conforme o arquivo escolhido.

In [ ]:
import time\nimport numpy as np\nimport pandas as pd\nimport seaborn as sns\nimport matplotlib.pyplot as plt\n\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\nfrom sklearn.decomposition import PCA\nfrom sklearn.feature_selection import SelectKBest, mutual_info_classif\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.metrics import (\n    accuracy_score, precision_recall_fscore_support, classification_report,\n    confusion_matrix, ConfusionMatrixDisplay\n)\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.svm import SVC\n\nfrom imblearn.over_sampling import SMOTE\nfrom imblearn.pipeline import Pipeline as ImbPipeline

In [ ]:
# Exemplo de carga (substitua pelo arquivo real):\nDATA_PATH = '../data/raw/seu_dataset.csv'\nTARGET_COLUMN = 'label'  # ajuste para o nome da coluna alvo\n\ndf = pd.read_csv(DATA_PATH)\ndisplay(df.head())\nprint(df.shape)

## 2) Pré-processamento e seleção/redução de características\nEstrutura para separar variáveis, aplicar codificação/escalonamento e reduzir dimensionalidade (PCA ou SelectKBest).

In [ ]:
X = df.drop(columns=[TARGET_COLUMN])\ny = df[TARGET_COLUMN]\n\nX_train, X_test, y_train, y_test = train_test_split(\n    X, y, test_size=0.2, random_state=42, stratify=y\n)\n\nnum_cols = X_train.select_dtypes(include=[np.number]).columns\ncat_cols = X_train.select_dtypes(exclude=[np.number]).columns\n\npreprocessor = ColumnTransformer(\n    transformers=[\n        ('num', StandardScaler(), num_cols),\n        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)\n    ],\n    remainder='drop'\n)\n\n# Opção A: Seleção de features\nn_features_to_select = min(100, max(1, int(0.8 * X_train.shape[1])))\nselector = SelectKBest(score_func=mutual_info_classif, k=n_features_to_select)\n\n# Opção B: Redução de dimensionalidade\n# pca = PCA(n_components=0.95, random_state=42)  # habilite esta opção ao testar redução por PCA


## 3) Tratamento de classes desbalanceadas\nUse SMOTE no treino ou `class_weight='balanced'` nos classificadores.

In [ ]:
smote = SMOTE(random_state=42)\n\n# Alternativa: class_weight='balanced' nos modelos (ver células seguintes).

## 4) Treinamento inicial: Random Forest e SVM\nRandom Forest para análise de importância de atributos e SVM para comparação de kernels.

In [ ]:
rf_model = RandomForestClassifier(\n    n_estimators=300,\n    random_state=42,\n    class_weight='balanced_subsample',\n    n_jobs=-1\n)\n\nrf_pipeline = ImbPipeline(steps=[\n    ('prep', preprocessor),\n    ('select', selector),  # alternativa: comente esta linha e habilite PCA\n    # ('pca', pca),         # opcional (escolher UMA estratégia)\n    ('smote', smote),\n    ('clf', rf_model)\n])\n\nstart_train = time.perf_counter()\nrf_pipeline.fit(X_train, y_train)\nrf_train_time = time.perf_counter() - start_train\n\nstart_infer = time.perf_counter()\nrf_pred = rf_pipeline.predict(X_test)\nrf_infer_time = time.perf_counter() - start_infer\n\nprint(f'RF treino (s): {rf_train_time:.4f}')\nprint(f'RF inferência (s): {rf_infer_time:.4f}')

In [ ]:
svm_results = {}\nfor kernel in ['linear', 'rbf', 'poly']:\n    svm_model = SVC(kernel=kernel, class_weight='balanced', random_state=42)\n\n    svm_pipeline = ImbPipeline(steps=[\n        ('prep', preprocessor),\n        ('select', selector),  # alternativa: comente esta linha e habilite PCA\n        # ('pca', pca),\n        ('smote', smote),\n        ('clf', svm_model)\n    ])\n\n    start_train = time.perf_counter()\n    svm_pipeline.fit(X_train, y_train)\n    train_time = time.perf_counter() - start_train\n\n    start_infer = time.perf_counter()\n    pred = svm_pipeline.predict(X_test)\n    infer_time = time.perf_counter() - start_infer\n\n    svm_results[kernel] = {\n        'pipeline': svm_pipeline,\n        'pred': pred,\n        'train_time': train_time,\n        'infer_time': infer_time\n    }\n\n    print(f'SVM ({kernel}) treino (s): {train_time:.4f}')\n    print(f'SVM ({kernel}) inferência (s): {infer_time:.4f}\
')

## 5) Avaliação de desempenho\nMétricas: Acurácia, Precisão, Recall, F1-score por classe e Matriz de Confusão.

In [ ]:
def avaliar_modelo(nome, y_true, y_pred):\n    acc = accuracy_score(y_true, y_pred)\n    pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)\n\n    print(f'===== {nome} =====')\n    print(f'Acurácia: {acc:.4f}')\n    print('Precisão por classe:', pr)\n    print('Recall por classe:', rc)\n    print('F1-score por classe:', f1)\n    print('\nRelatório completo:')\n    print(classification_report(y_true, y_pred, zero_division=0))\n\n    cm = confusion_matrix(y_true, y_pred)\n    disp = ConfusionMatrixDisplay(confusion_matrix=cm)\n    disp.plot(cmap='Blues', values_format='d')\n    plt.title(f'Matriz de Confusão - {nome}')\n    plt.show()

In [ ]:
avaliar_modelo('Random Forest', y_test, rf_pred)\n\nfor kernel, result in svm_results.items():\n    avaliar_modelo(f'SVM ({kernel})', y_test, result['pred'])